# SIADS 699 — Dota 2 Movement Feature Engineering

**Project:** Predicting Dota 2 Player Skill via Spatio-Temporal Movement Data

This notebook builds the **L1** (per-player, per-timestep) and **L2** (player-match) feature tables from the raw positions.

### Pipeline overview
1. Load data
2. Apply team-relative coordinate transform
3. Build death mask
4. Compute basic L1 features
5. Aggregate to L2 (player-match level)
6. Attach labels & metadata
7. Save results + quick inspection


Get Data

In [ ]:
from google.colab import auth; auth.authenticate_user()
!gsutil -m cp -r gs://siads699-dota/dataset/v1/ .

Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Optional plotting
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries loaded")

## 1. Paths & Constants

In [ ]:
# Colab path (data downloaded to /content/v1/)
DATA_DIR = Path("/content/v1")                        # positions.parquet, players.csv, matches.csv
OUT_DIR  = Path("/content/features")                  # output location
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fountain coordinates measured from the map (from README — do not change)
R = np.array([9515.0, 9975.0])      # Radiant fountain
D = np.array([23451.0, 22746.0])    # Dire fountain
AXIS = D - R
L = float(np.linalg.norm(AXIS))
U = AXIS / L                        # unit vector along the main axis
N = np.array([-U[1], U[0]])         # perpendicular (lateral)

DIRE_SLOTS = {128, 129, 130, 131, 132}

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)
print("Axis length L:", round(L, 1))

## 2. Team-relative coordinate transform

Raw `x, y` cannot be used directly. Radiant spawns at low coordinates and Dire at high ones, so the same number means the opposite thing for the two teams. A model would learn which side you were on before it learns anything about skill.

We convert to:
- `own_depth`  : 0 = own fountain, 1 = enemy fountain
- `own_lateral`: signed distance from the mid diagonal (mirrored for Dire)


In [ ]:
def to_team_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Raw x,y → own_depth and own_lateral. MUST be run before any feature work."""
    v = df[["x", "y"]].to_numpy(dtype=np.float64) - R
    depth   = (v @ U) / L
    lateral = v @ N
    dire = df["player_slot"].isin(DIRE_SLOTS).to_numpy()

    out = df.copy()
    out["own_depth"]   = np.where(dire, 1.0 - depth, depth)
    out["own_lateral"] = np.where(dire, -lateral, lateral)
    return out

print("to_team_frame() defined")

## 3. Load data

In [ ]:
print("Loading metadata...")
players = pd.read_csv(DATA_DIR / "players.csv")
matches = pd.read_csv(DATA_DIR / "matches.csv")

print("Loading positions (large file)...")
positions = pd.read_parquet(DATA_DIR / "positions.parquet")

print(f"positions : {positions.shape}")
print(f"players   : {players.shape}")
print(f"matches   : {matches.shape}")
print(f"unique matches in positions: {positions['match_id'].nunique()}")
print(f"player_slot values: {sorted(positions['player_slot'].unique())}")

In [ ]:
positions.head(3)

In [ ]:
players.head(3)

In [ ]:
matches["run_id"].value_counts()

## 4. Validate the coordinate transform

We take one match and check that after the transform:
- Both teams start near `own_depth ≈ 0` at the beginning of the game
- The two teams look roughly symmetric


In [ ]:
# Pick a short match for quick visual check
sample_match_id = matches.loc[matches["duration_s"].between(1800, 2200), "match_id"].iloc[0]
print("Sample match_id:", sample_match_id)

pos_sample = positions[positions["match_id"] == str(sample_match_id)].copy()
pos_sample = to_team_frame(pos_sample)

print("\nown_depth range :", round(pos_sample["own_depth"].min(), 3), "→", round(pos_sample["own_depth"].max(), 3))
print("own_lateral range:", round(pos_sample["own_lateral"].min(), 1), "→", round(pos_sample["own_lateral"].max(), 1))

# Early-game depth by team
early = pos_sample[pos_sample["sec"] < 60]
print("\nMean own_depth in first 60s:")
print(early.groupby(early["player_slot"].isin(DIRE_SLOTS))["own_depth"].mean())

In [ ]:
# Optional: quick scatter of early positions in team frame
fig, ax = plt.subplots(figsize=(6, 6))
for is_dire, color, label in [(False, "C0", "Radiant"), (True, "C1", "Dire")]:
    mask = early["player_slot"].isin(DIRE_SLOTS) == is_dire
    ax.scatter(early.loc[mask, "own_lateral"], early.loc[mask, "own_depth"],
               s=8, alpha=0.5, c=color, label=label)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.axvline(0, color="gray", ls="--", lw=0.8)
ax.set_xlabel("own_lateral")
ax.set_ylabel("own_depth")
ax.set_title(f"First 60s — match {sample_match_id}")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 5. Process one match → L2 features

We process match-by-match to keep memory usage low.

For each match we:
1. Apply the team-relative transform
2. Build a simple death mask (player teleports back near fountain)
3. Compute movement statistics only on live (non-dead) time
4. Aggregate to one row per player


In [ ]:
def assign_phase(sec: float) -> str:
    if sec < 600:
        return "laning"   # 0-10 min
    if sec < 1500:
        return "mid"      # 10-25 min
    return "late"


def process_one_match(pos: pd.DataFrame) -> pd.DataFrame:
    """
    Given positions for a single match, return one row per player_slot with L2 features.
    """
    pos = to_team_frame(pos)
    pos = pos.sort_values(["player_slot", "sec"])

    # --- death mask (simple heuristic) ---
    pos["prev_depth"] = pos.groupby("player_slot")["own_depth"].shift(1)
    pos["is_dead"] = (
        (pos["own_depth"] < 0.08) & (pos["prev_depth"] > 0.25)
    ).fillna(False)

    DEAD_WINDOW = 8.0  # seconds after detected death to keep masked
    pos["dead_mask"] = False
    for slot, grp in pos.groupby("player_slot"):
        dead_starts = grp.loc[grp["is_dead"], "sec"].values
        if len(dead_starts) == 0:
            continue
        secs = grp["sec"].values
        mask = np.zeros(len(grp), dtype=bool)
        for t0 in dead_starts:
            mask |= (secs >= t0) & (secs < t0 + DEAD_WINDOW)
        pos.loc[grp.index, "dead_mask"] = mask

    # --- movement features ---
    pos["dt"] = pos.groupby("player_slot")["sec"].diff().fillna(2.0)
    pos["dx"] = pos.groupby("player_slot")["own_depth"].diff()
    pos["dy"] = pos.groupby("player_slot")["own_lateral"].diff()
    pos["dist"] = np.sqrt(pos["dx"]**2 + pos["dy"]**2)
    pos["speed"] = pos["dist"] / pos["dt"].clip(lower=0.5)
    pos["is_idle"] = (pos["speed"] < 50) & (~pos["dead_mask"])

    # rough zone (can be improved with proper polygons later)
    pos["zone"] = pd.cut(
        pos["own_lateral"],
        bins=[-np.inf, -2000, 2000, np.inf],
        labels=["top", "mid", "bottom"]
    )
    pos["phase"] = pos["sec"].map(assign_phase)

    # only live rows for most stats
    live = pos.loc[~pos["dead_mask"]].copy()
    if live.empty:
        return pd.DataFrame()

    agg = live.groupby("player_slot").agg(
        n_samples        = ("sec", "count"),
        total_dist       = ("dist", "sum"),
        mean_speed       = ("speed", "mean"),
        median_speed     = ("speed", "median"),
        idle_share       = ("is_idle", "mean"),
        mean_depth       = ("own_depth", "mean"),
        median_depth     = ("own_depth", "median"),
        max_depth        = ("own_depth", "max"),
        depth_std        = ("own_depth", "std"),
        lateral_std      = ("own_lateral", "std"),
        mean_abs_lateral = ("own_lateral", lambda s: np.abs(s).mean()),
        first_sec        = ("sec", "min"),
        last_sec         = ("sec", "max"),
    ).reset_index()

    agg["duration"] = agg["last_sec"] - agg["first_sec"]
    agg["path_efficiency"] = agg["total_dist"] / agg["duration"].clip(lower=1.0)

    # zone × phase occupancy shares
    zp = (
        live.groupby(["player_slot", "zone", "phase"])
        .size()
        .unstack(level=["zone", "phase"], fill_value=0)
    )
    zp.columns = [f"share_{z}_{p}" for z, p in zp.columns]
    zp = zp.div(zp.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    zp = zp.reset_index()

    out = agg.merge(zp, on="player_slot", how="left")
    out["match_id"] = pos["match_id"].iloc[0]
    return out

print("process_one_match() defined")

## 6. Build full L2 table

This loops over all 533 matches. It takes a couple of minutes.

In [ ]:
match_ids = positions["match_id"].unique()
print(f"{len(match_ids)} matches to process")

rows = []
for i, mid in enumerate(match_ids):
    if (i + 1) % 50 == 0 or i == 0:
        print(f"  processing match {i+1}/{len(match_ids)} ...")
    pos_m = positions[positions["match_id"] == mid]
    feat = process_one_match(pos_m)
    if not feat.empty:
        rows.append(feat)

print("Concatenating...")
l2 = pd.concat(rows, ignore_index=True)

# match_id is str in parquet, int in the CSVs
l2["match_id"] = l2["match_id"].astype(int)

print(f"L2 shape before joins: {l2.shape}")

## 7. Attach labels and match metadata

In [ ]:
l2 = l2.merge(
    players[["match_id", "player_slot", "account_id", "hero_id", "rank_tier", "leaver_status"]],
    on=["match_id", "player_slot"],
    how="left"
)
l2 = l2.merge(
    matches[["match_id", "skill_label", "avg_rank_tier", "region", "duration_s", "run_id"]],
    on="match_id",
    how="left"
)

print(f"Final L2 shape: {l2.shape}")
print(f"Players with rank_tier: {l2['rank_tier'].notna().sum()}")
l2.head()

## 8. Quick inspection

In [ ]:
l2[["mean_depth", "idle_share", "path_efficiency", "rank_tier"]].describe()

In [ ]:
# Distribution of key features by skill_label
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["mean_depth", "idle_share", "path_efficiency"]):
    for label, color in [("low", "C0"), ("high", "C1")]:
        subset = l2.loc[l2["skill_label"] == label, col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation with rank_tier (simple check)
numeric_cols = ["mean_depth", "idle_share", "path_efficiency", "mean_speed",
                "depth_std", "lateral_std", "max_depth"]
print(l2[numeric_cols + ["rank_tier"]].corr()["rank_tier"].sort_values(ascending=False))

## 9. Save outputs

In [ ]:
l2_path_csv = OUT_DIR / "l2_player_match.csv"
l2_path_parquet = OUT_DIR / "l2_player_match.parquet"

l2.to_csv(l2_path_csv, index=False)
try:
    l2.to_parquet(l2_path_parquet, index=False)
    print(f"Saved parquet → {l2_path_parquet}")
except Exception as e:
    print(f"Parquet save failed ({e}), CSV is still available")

print(f"Saved CSV     → {l2_path_csv}")
print(f"Shape: {l2.shape}")

# small sample for quick looks
sample = l2.sample(min(30, len(l2)), random_state=42)
sample.to_csv(OUT_DIR / "l2_sample.csv", index=False)
print(f"Saved sample  → {OUT_DIR / 'l2_sample.csv'}")

## 10. Time-thirds × Lane analysis (high vs low rank)

The main L2 features use fixed clock phases (0–10 / 10–25 / 25+ min).
Here we add a complementary view that splits **each match into equal thirds** of its own duration:

- **early** = first third of the match
- **mid**   = middle third
- **late**  = last third

We also keep the three **lanes** (top / mid / bottom) from `own_lateral`.
This lets us compare how high-rank and low-rank players move in different stages of the game and in different lanes.

Zone rule (same as in `process_one_match`):
- `own_lateral < -2000` → **top**
- `-2000 ≤ own_lateral ≤ 2000` → **mid**
- `own_lateral > 2000` → **bottom**


In [ ]:
def assign_phase_third(sec, match_duration):
    """Split a match into equal thirds of its own length."""
    if match_duration is None or match_duration <= 0:
        return "mid"
    t1 = match_duration / 3.0
    t2 = 2.0 * match_duration / 3.0
    if sec < t1:
        return "early"
    if sec < t2:
        return "mid"
    return "late"


def build_l1_with_thirds(pos, matches_df):
    """
    Enrich positions with:
      - team-relative coordinates
      - zone (top / mid / bottom)
      - phase_third (early / mid / late)
      - death mask, speed, idle flag
    """
    # Ensure match_id in pos is int to match matches_df
    pos["match_id"] = pos["match_id"].astype(int)

    dur = matches_df.set_index("match_id")["duration_s"].to_dict()

    pos = to_team_frame(pos.copy())
    pos = pos.sort_values(["match_id", "player_slot", "sec"])

    # lanes
    pos["zone"] = pd.cut(
        pos["own_lateral"],
        bins=[-np.inf, -2000, 2000, np.inf],
        labels=["top", "mid", "bottom"],
    )

    # equal thirds of each match
    pos["match_duration"] = pos["match_id"].map(dur)
    pos["phase_third"] = [
        assign_phase_third(s, d) for s, d in zip(pos["sec"], pos["match_duration"])
    ]

    # simple death mask (teleport back near fountain)
    pos["prev_depth"] = pos.groupby(["match_id", "player_slot"])["own_depth"].shift(1)
    pos["is_dead"] = ((pos["own_depth"] < 0.08) & (pos["prev_depth"] > 0.25)).fillna(False)

    DEAD_WINDOW = 8.0
    pos["dead_mask"] = False
    for (mid, slot), grp in pos.groupby(["match_id", "player_slot"]):
        dead_starts = grp.loc[grp["is_dead"], "sec"].values
        if len(dead_starts) == 0:
            continue
        secs = grp["sec"].values
        mask = np.zeros(len(grp), dtype=bool)
        for t0 in dead_starts:
            mask |= (secs >= t0) & (secs < t0 + DEAD_WINDOW)
        pos.loc[grp.index, "dead_mask"] = mask

    # movement basics
    g = pos.groupby(["match_id", "player_slot"])
    pos["dt"] = g["sec"].diff().fillna(2.0)
    pos["dx"] = g["own_depth"].diff()
    pos["dy"] = g["own_lateral"].diff()
    pos["dist"] = np.sqrt(pos["dx"]**2 + pos["dy"]**2)
    pos["speed"] = pos["dist"] / pos["dt"].clip(lower=0.5)
    pos["is_idle"] = (pos["speed"] < 50) & (~pos["dead_mask"])

    return pos


print("Building L1 with equal-thirds phases and lanes...")
print("(A couple of minutes on 7.2M rows — go pet a cat or something)")

l1 = build_l1_with_thirds(positions, matches)

l1 = l1.merge(
    players[["match_id", "player_slot", "rank_tier"]],
    on=["match_id", "player_slot"],
    how="left",
)
l1 = l1.merge(
    matches[["match_id", "skill_label"]],
    on="match_id",
    how="left",
)

print(f"L1 shape: {l1.shape}")
print("phase_third counts:\n", l1["phase_third"].value_counts())
print("zone counts:\n", l1["zone"].value_counts())
l1[["match_id", "player_slot", "sec", "own_depth", "zone", "phase_third", "skill_label"]].head(6)


In [ ]:
# Live rows only for movement stats
live = l1.loc[~l1["dead_mask"]].copy()

# per player × match × phase × zone
player_bucket = (
    live.groupby(
        ["match_id", "player_slot", "skill_label", "rank_tier", "phase_third", "zone"],
        observed=True,
    )
    .agg(
        n_samples=("sec", "size"),
        mean_depth=("own_depth", "mean"),
        mean_speed=("speed", "mean"),
        idle_share=("is_idle", "mean"),
        total_dist=("dist", "sum"),
        mean_abs_lat=("own_lateral", lambda s: np.abs(s).mean()),
    )
    .reset_index()
)

print("Player-bucket shape:", player_bucket.shape)

# high vs low average behaviour
summary = (
    player_bucket.groupby(["skill_label", "phase_third", "zone"], observed=True)
    .agg(
        n_players=("player_slot", "count"),
        mean_depth=("mean_depth", "mean"),
        mean_speed=("mean_speed", "mean"),
        idle_share=("idle_share", "mean"),
        mean_abs_lat=("mean_abs_lat", "mean"),
    )
    .reset_index()
    .sort_values(["phase_third", "zone", "skill_label"])
)

print("\n=== High vs Low by phase_third × zone ===")
display(summary)



In [ ]:
# What fraction of live time do high vs low players spend in each lane, by phase?
time_share = (
    live.groupby(["skill_label", "phase_third", "zone"], observed=True)
    .size()
    .rename("n_samples")
    .reset_index()
)

time_share["lane_share"] = (
    time_share.groupby(["skill_label", "phase_third"])["n_samples"]
    .transform(lambda s: s / s.sum())
)

print("Lane occupancy (share of live samples) by skill and phase:")
pivot = time_share.pivot_table(
    index=["phase_third", "skill_label"],
    columns="zone",
    values="lane_share",
    fill_value=0,
)
display(pivot.round(3))

# handy constants for the plots below
phases = ["early", "mid", "late"]
lanes = ["top", "mid", "bottom"]
x = np.arange(len(lanes))
width = 0.35



In [ ]:
player_bucket.to_csv(OUT_DIR / "player_phase_zone_buckets.csv", index=False)
summary.to_csv(OUT_DIR / "skill_phase_zone_summary.csv", index=False)
time_share.to_csv(OUT_DIR / "lane_time_share_by_skill_phase.csv", index=False)

print("Saved comparison tables to", OUT_DIR)
print("You can now compare high vs low on lane time, aggression, idle time, and map location.")



**bold text**## Modeling (XGBoost + ablations + SHAP)

We train a few sensible movement feature sets instead of every possible subset.
Goal: see whether movement predicts skill, and which feature families actually help.


In [ ]:

# Modeling: baseline + sensible feature ablations and No, we are NOT trying 16 million combinations. Computers have feelings too.)

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
import matplotlib.pyplot as plt

print("Libraries ready. Time to see if movement actually means anything.")

# --- 1. Prep once ---
data = l2.dropna(subset=["skill_label"]).copy()
data["target"] = data["skill_label"].map({"low": 0, "high": 1})

ALWAYS_DROP = [
    "match_id", "player_slot", "account_id", "skill_label", "run_id",
    "target", "region", "leaver_status", "avg_rank_tier", "rank_tier", "hero_id",
]

SPEED_COLS = [c for c in data.columns if "speed" in c.lower()]
ZONE_SHARE_COLS = [c for c in data.columns if c.startswith("share_")]
CORE_MOVE = [
    c for c in data.columns
    if c not in ALWAYS_DROP
    and c not in ZONE_SHARE_COLS
    and not c.startswith("first_")
    and not c.startswith("last_")
]

print(f"Rows with labels : {len(data)}")
print(f"Speed-ish cols   : {SPEED_COLS}")
print(f"Zone-share cols  : {len(ZONE_SHARE_COLS)}")

def train_and_score(feature_cols, label="model"):
    """Train one XGBoost and return metrics. Small and boring on purpose."""
    cols = [c for c in feature_cols if c in data.columns]
    if not cols:
        return {"label": label, "n_features": 0, "accuracy": np.nan, "auc": np.nan, "f1": np.nan}

    X = data[cols].fillna(data[cols].median())
    y = data["target"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "label": label,
        "n_features": len(cols),
        "accuracy": accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba),
        "f1": f1_score(y_test, pred),
        "model": model,
        "X_test": X_test,
        "y_test": y_test,
    }

# --- 2. Feature sets that match the project story ---
all_move = [c for c in data.columns if c not in ALWAYS_DROP]

feature_sets = {
    "All movement features": all_move,
    "No speed features": [c for c in all_move if c not in SPEED_COLS],
    "No zone-share features": [c for c in all_move if c not in ZONE_SHARE_COLS],
    "Core trajectory only": [c for c in CORE_MOVE if c in all_move],
    "Depth + path + idle only": [
        c for c in all_move
        if ("depth" in c.lower() or "path" in c.lower() or "idle" in c.lower())
    ],
}

print("\nTraining a handful of models (about 1–2 minutes, not 2 hours)...")
results, models = [], {}

for name, cols in feature_sets.items():
    print(f"  → {name} ({len(cols)} features)")
    out = train_and_score(cols, label=name)
    results.append({k: out[k] for k in ["label", "n_features", "accuracy", "auc", "f1"]})
    models[name] = out

results_df = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
print("\n" + "=" * 60)
print("ABLATION RESULTS (sorted by AUC)")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)

best_name = results_df.loc[0, "label"]
print(f"\nBest set: '{best_name}'  (AUC = {results_df.loc[0, 'auc']:.4f})")
print("Note: simple random split here. For the report, also run grouped CV by match_id.")

best = models[best_name]
print("\nClassification report (best feature set):")
print(classification_report(
    best["y_test"],
    (best["model"].predict_proba(best["X_test"])[:, 1] >= 0.5).astype(int),
    target_names=["Low Skill", "High Skill"],
))

# --- 3. SHAP on the winner ---
print("SHAP time — this is the part that makes the report look clever.")
explainer = shap.TreeExplainer(best["model"])
shap_values = explainer.shap_values(best["X_test"])

plt.figure()
shap.summary_plot(shap_values, best["X_test"], plot_type="bar", show=False)
plt.title(f"SHAP importance — {best_name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_shap_bar.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure()
shap.summary_plot(shap_values, best["X_test"], plot_type="dot", show=False)
plt.title(f"SHAP beeswarm — {best_name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_shap_beeswarm.png", dpi=160, bbox_inches="tight")
plt.show()

print("Saved SHAP figures →", OUT_DIR)
print("Done. Movement model + ablations + SHAP, CPU still friends with us.")



## 11. Report visualizations

Clean plots comparing **high** vs **low** rank players across:
- the three equal match thirds (early / mid / late)
- the three lanes (top / mid / bottom)

These are designed to drop straight into the final report.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Make sure we have the summary tables from the previous section
assert "time_share" in dir() or "time_share" in globals(), "Run the Phase-thirds cells first"

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
phases = ["early", "mid", "late"]
lanes  = ["top", "mid", "bottom"]
x = np.arange(len(lanes))
width = 0.35

for ax, phase in zip(axes, phases):
    for i, (skill, color) in enumerate([("low", "#4C72B0"), ("high", "#DD8452")]):
        sub = time_share[(time_share["phase_third"] == phase) & (time_share["skill_label"] == skill)]
        # ensure order
        sub = sub.set_index("zone").reindex(lanes).fillna(0)
        ax.bar(x + i*width, sub["lane_share"].values, width=width, label=skill, color=color, alpha=0.9)
    ax.set_xticks(x + width/2)
    ax.set_xticklabels([l.capitalize() for l in lanes])
    ax.set_title(f"{phase.capitalize()} third of match")
    ax.set_ylim(0, 0.65)
    if ax is axes[0]:
        ax.set_ylabel("Share of live time in lane")
        ax.legend(frameon=False)

fig.suptitle("Lane occupancy: High vs Low rank players", fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_lane_occupancy.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_lane_occupancy.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, phase in zip(axes, phases):
    sub = summary[summary["phase_third"] == phase]
    for i, (skill, color) in enumerate([("low", "#4C72B0"), ("high", "#DD8452")]):
        s = sub[sub["skill_label"] == skill].set_index("zone").reindex(lanes)
        ax.bar(x + i*width, s["mean_depth"].values, width=width, label=skill, color=color, alpha=0.9)
    ax.set_xticks(x + width/2)
    ax.set_xticklabels([l.capitalize() for l in lanes])
    ax.set_title(f"{phase.capitalize()} third")
    if ax is axes[0]:
        ax.set_ylabel("Mean own_depth\n(0 = fountain, 1 = enemy)")
        ax.legend(frameon=False)

fig.suptitle("Aggression (mean depth) by lane and match third", fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_mean_depth.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_mean_depth.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, phase in zip(axes, phases):
    sub = summary[summary["phase_third"] == phase]
    for i, (skill, color) in enumerate([("low", "#4C72B0"), ("high", "#DD8452")]):
        s = sub[sub["skill_label"] == skill].set_index("zone").reindex(lanes)
        ax.bar(x + i*width, s["idle_share"].values, width=width, label=skill, color=color, alpha=0.9)
    ax.set_xticks(x + width/2)
    ax.set_xticklabels([l.capitalize() for l in lanes])
    ax.set_title(f"{phase.capitalize()} third")
    if ax is axes[0]:
        ax.set_ylabel("Idle share (speed < 50)")
        ax.legend(frameon=False)

fig.suptitle("Idle time share by lane and match third", fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_idle_share.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_idle_share.png")


In [ ]:
# Difference heatmap: high − low mean depth (positive = high-rank more aggressive)
pivot_high = summary[summary["skill_label"] == "high"].pivot(index="phase_third", columns="zone", values="mean_depth")
pivot_low  = summary[summary["skill_label"] == "low"].pivot(index="phase_third", columns="zone", values="mean_depth")

# align order
pivot_high = pivot_high.reindex(index=phases, columns=lanes)
pivot_low  = pivot_low.reindex(index=phases, columns=lanes)
diff = pivot_high - pivot_low

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

for ax, (data, title, cmap, center) in zip(axes, [
    (pivot_low,  "Low rank — mean depth",  "Blues",   None),
    (pivot_high, "High rank — mean depth", "Oranges", None),
    (diff,       "High − Low (aggression gap)", "RdBu_r", 0),
]):
    im = ax.imshow(data.values, cmap=cmap, aspect="auto", vmin=None if center is None else -0.08, vmax=None if center is None else 0.08)
    ax.set_xticks(range(3)); ax.set_xticklabels([l.capitalize() for l in lanes])
    ax.set_yticks(range(3)); ax.set_yticklabels([p.capitalize() for p in phases])
    ax.set_title(title, fontsize=11)
    for i in range(3):
        for j in range(3):
            val = data.values[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                    color="white" if (center is not None and abs(val) > 0.04) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Mean depth (aggression) heatmaps", y=1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_depth_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_depth_heatmap.png")


In [ ]:
# Density of player positions in the team-relative frame during the EARLY third
# Sample to keep it fast
early = l1[(l1["phase_third"] == "early") & (~l1["dead_mask"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

for ax, skill, cmap in zip(axes, ["low", "high"], ["Blues", "Oranges"]):
    sub = early[early["skill_label"] == skill]
    # subsample for speed / clarity
    if len(sub) > 80000:
        sub = sub.sample(80000, random_state=42)
    hb = ax.hexbin(sub["own_lateral"], sub["own_depth"], gridsize=45, cmap=cmap, mincnt=3, bins="log")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8, alpha=0.7)   # river-ish line
    ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.7)      # mid diagonal
    ax.set_title(f"{skill.capitalize()} rank — early third")
    ax.set_xlabel("own_lateral (← top | bottom →)")
    if ax is axes[0]:
        ax.set_ylabel("own_depth (0 = own fountain)")
    ax.set_xlim(-9000, 9000)
    ax.set_ylim(-0.05, 1.05)
    fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Where players spend time early (team-relative coordinates)", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_density_early.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_density_early.png")


In [ ]:
late = l1[(l1["phase_third"] == "late") & (~l1["dead_mask"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

for ax, skill, cmap in zip(axes, ["low", "high"], ["Blues", "Oranges"]):
    sub = late[late["skill_label"] == skill]
    if len(sub) > 80000:
        sub = sub.sample(80000, random_state=42)
    hb = ax.hexbin(sub["own_lateral"], sub["own_depth"], gridsize=45, cmap=cmap, mincnt=3, bins="log")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8, alpha=0.7)
    ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.7)
    ax.set_title(f"{skill.capitalize()} rank — late third")
    ax.set_xlabel("own_lateral (← top | bottom →)")
    if ax is axes[0]:
        ax.set_ylabel("own_depth (0 = own fountain)")
    ax.set_xlim(-9000, 9000)
    ax.set_ylim(-0.05, 1.05)
    fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Where players spend time late (team-relative coordinates)", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_density_late.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_density_late.png")


In [ ]:
# One compact multi-panel figure useful as a single report graphic
fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(2, 2, hspace=0.32, wspace=0.28)

# (0,0) Lane share early
ax0 = fig.add_subplot(gs[0, 0])
for i, (skill, color) in enumerate([("low", "#4C72B0"), ("high", "#DD8452")]):
    sub = time_share[(time_share["phase_third"] == "early") & (time_share["skill_label"] == skill)]
    sub = sub.set_index("zone").reindex(lanes).fillna(0)
    ax0.bar(x + i*width, sub["lane_share"], width=width, label=skill, color=color)
ax0.set_xticks(x + width/2); ax0.set_xticklabels([l.capitalize() for l in lanes])
ax0.set_ylabel("Lane share"); ax0.set_title("Early — lane occupancy"); ax0.legend(frameon=False, fontsize=9)

# (0,1) Depth gap heatmap
ax1 = fig.add_subplot(gs[0, 1])
im = ax1.imshow(diff.values, cmap="RdBu_r", aspect="auto", vmin=-0.06, vmax=0.06)
ax1.set_xticks(range(3)); ax1.set_xticklabels([l.capitalize() for l in lanes])
ax1.set_yticks(range(3)); ax1.set_yticklabels([p.capitalize() for p in phases])
ax1.set_title("Aggression gap (High − Low)")
for i in range(3):
    for j in range(3):
        ax1.text(j, i, f"{diff.values[i,j]:.3f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# (1,0) Idle share late
ax2 = fig.add_subplot(gs[1, 0])
sub = summary[summary["phase_third"] == "late"]
for i, (skill, color) in enumerate([("low", "#4C72B0"), ("high", "#DD8452")]):
    s = sub[sub["skill_label"] == skill].set_index("zone").reindex(lanes)
    ax2.bar(x + i*width, s["idle_share"], width=width, label=skill, color=color)
ax2.set_xticks(x + width/2); ax2.set_xticklabels([l.capitalize() for l in lanes])
ax2.set_ylabel("Idle share"); ax2.set_title("Late — idle share"); ax2.legend(frameon=False, fontsize=9)

# (1,1) Mean speed by phase (overall, not by lane)
ax3 = fig.add_subplot(gs[1, 1])
speed_by_phase = (
    player_bucket.groupby(["skill_label", "phase_third"])["mean_speed"]
    .mean().reset_index()
)
for skill, color in [("low", "#4C72B0"), ("high", "#DD8452")]:
    s = speed_by_phase[speed_by_phase["skill_label"] == skill].set_index("phase_third").reindex(phases)
    ax3.plot(phases, s["mean_speed"], marker="o", label=skill, color=color, lw=2)
ax3.set_ylabel("Mean speed"); ax3.set_title("Movement speed across match thirds")
ax3.legend(frameon=False, fontsize=9); ax3.set_xlabel("Match third")

fig.suptitle("Movement differences between High and Low rank players", fontsize=14, y=0.98)
plt.savefig(OUT_DIR / "fig_report_summary.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_report_summary.png")
print("\nAll report figures are in:", OUT_DIR)


## 13. Path efficiency + map path visualizations

Raw averages of depth/idle were small. Here we check:
1. Whether **path_efficiency** (and friends) separate ranks any better
2. Whether high and low ranks **use the map differently** (density + example trajectories)

These plots are meant for the report: "do different ranks take different paths?"


In [ ]:
# --- Is path_efficiency more promising than mean_depth? ---
# Uses the cleaned individual-rank table from section 12 if available, else falls back to l2.

src = l2_ind if "l2_ind" in dir() or "l2_ind" in globals() else l2.copy()
if "indiv_label" not in src.columns:
    src = src.dropna(subset=["rank_tier"]).copy()
    if "leaver_status" in src.columns:
        src = src[src["leaver_status"] == 0]
    src["indiv_label"] = np.where(src["rank_tier"] >= 50, "high", "low")
    src["band"] = np.where(src["rank_tier"] >= 60, "high60+",
                    np.where(src["rank_tier"] <= 30, "low30-", "mid"))

check_cols = [c for c in [
    "path_efficiency", "mean_depth", "mean_speed", "idle_share",
    "depth_std", "lateral_std", "max_depth", "total_dist"
] if c in src.columns]

print("Feature means by individual rank:")
print(src.groupby("indiv_label")[check_cols].mean().round(4).to_string())

print("\nStrict bands (<=30 vs >=60):")
strict = src[src["band"].isin(["low30-", "high60+"])]
print(strict.groupby("band")[check_cols].mean().round(4).to_string())

# effect size-ish: (high - low) / pooled std
rows = []
for col in check_cols:
    a = src.loc[src["indiv_label"] == "high", col].dropna()
    b = src.loc[src["indiv_label"] == "low", col].dropna()
    if len(a) < 10 or len(b) < 10:
        continue
    pooled = np.sqrt((a.var() + b.var()) / 2)
    gap = a.mean() - b.mean()
    rows.append({"feature": col, "high_mean": a.mean(), "low_mean": b.mean(),
                 "gap": gap, "pooled_std": pooled,
                 "gap_over_std": gap / pooled if pooled > 0 else np.nan})
eff = pd.DataFrame(rows).sort_values("gap_over_std", key=lambda s: s.abs(), ascending=False)
print("\nRanked by |gap| / std (bigger = more separation):")
print(eff.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#DD8452" if g > 0 else "#4C72B0" for g in eff["gap_over_std"]]
ax.barh(eff["feature"], eff["gap_over_std"], color=colors)
ax.axvline(0, color="gray", lw=1)
ax.set_xlabel("(high - low) / pooled std")
ax.set_title("Which L2 features separate individual rank at all?")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_feature_separation.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_feature_separation.png")


In [ ]:
# --- Map density: high vs low paths in team-relative coordinates ---
# Needs l1 from section 10. If you skipped that section, rebuild a light version here.

if "l1" not in globals() and "l1" not in dir():
    print("l1 not found — building a lightweight version for plots (sampled matches)...")
    # sample matches to keep Colab memory happy
    rng = np.random.default_rng(42)
    sample_ids = rng.choice(positions["match_id"].unique(), size=min(120, positions["match_id"].nunique()), replace=False)
    pos_s = positions[positions["match_id"].isin(sample_ids)].copy()
    pos_s = to_team_frame(pos_s)
    dur = matches.set_index("match_id")["duration_s"].to_dict()
    md = pos_s["match_id"].map(dur)
    sec = pos_s["sec"]
    phase = np.full(len(pos_s), "mid", dtype=object)
    valid = md.notna() & (md > 0)
    phase[valid & (sec < md/3)] = "early"
    phase[valid & (sec >= 2*md/3)] = "late"
    pos_s["phase_third"] = phase
    pos_s = pos_s.merge(players[["match_id", "player_slot", "rank_tier", "leaver_status"]],
                        on=["match_id", "player_slot"], how="left")
    pos_s = pos_s.dropna(subset=["rank_tier"])
    pos_s = pos_s[pos_s["leaver_status"] == 0]
    pos_s["indiv_label"] = np.where(pos_s["rank_tier"] >= 50, "high", "low")
    l1_plot = pos_s
else:
    l1_plot = l1.copy()
    if "rank_tier" not in l1_plot.columns:
        l1_plot = l1_plot.merge(
            players[["match_id", "player_slot", "rank_tier", "leaver_status"]],
            on=["match_id", "player_slot"], how="left"
        )
    l1_plot = l1_plot.dropna(subset=["rank_tier"])
    if "leaver_status" in l1_plot.columns:
        l1_plot = l1_plot[l1_plot["leaver_status"] == 0]
    l1_plot["indiv_label"] = np.where(l1_plot["rank_tier"] >= 50, "high", "low")

# live-ish points only
if "dead_mask" in l1_plot.columns:
    l1_plot = l1_plot.loc[~l1_plot["dead_mask"]].copy()
else:
    l1_plot = l1_plot.loc[l1_plot["own_depth"] >= 0.05].copy()

print("Plot rows:", len(l1_plot))
print(l1_plot["indiv_label"].value_counts())


In [ ]:
# Density comparison: where high vs low players actually are (early + late)

def density_pair(df, phase, fname):
    sub = df[df["phase_third"] == phase] if "phase_third" in df.columns else df
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
    for ax, skill, cmap in zip(axes, ["low", "high"], ["Blues", "Oranges"]):
        s = sub[sub["indiv_label"] == skill]
        if len(s) > 70000:
            s = s.sample(70000, random_state=42)
        hb = ax.hexbin(s["own_lateral"], s["own_depth"], gridsize=40, cmap=cmap, mincnt=2, bins="log")
        ax.axhline(0.5, color="gray", ls="--", lw=0.8, alpha=0.7)
        ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.7)
        ax.set_title(f"{skill} rank — {phase}")
        ax.set_xlabel("own_lateral (lanes)")
        ax.set_xlim(-9000, 9000)
        ax.set_ylim(-0.05, 1.05)
        fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
    axes[0].set_ylabel("own_depth (0 = fountain, 1 = enemy)")
    fig.suptitle(f"Map occupancy by rank — {phase} third", y=1.02)
    plt.tight_layout()
    path = OUT_DIR / fname
    plt.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()
    print("Saved →", path)

density_pair(l1_plot, "early", "fig_paths_density_early.png")
density_pair(l1_plot, "late", "fig_paths_density_late.png")


In [ ]:
# Difference density: high minus low occupancy (where do high ranks go MORE?)
# Positive = high ranks spend relatively more time there.

def rank_density_grid(df, phase, bins=35):
    sub = df[df["phase_third"] == phase].copy() if "phase_third" in df.columns else df.copy()
    # same limits for both
    xlim = (-8000, 8000)
    ylim = (0.0, 1.0)
    high = sub[sub["indiv_label"] == "high"]
    low = sub[sub["indiv_label"] == "low"]
    if len(high) > 60000:
        high = high.sample(60000, random_state=0)
    if len(low) > 60000:
        low = low.sample(60000, random_state=0)
    h_hi, xedges, yedges = np.histogram2d(
        high["own_lateral"], high["own_depth"], bins=bins, range=[xlim, ylim], density=True
    )
    h_lo, _, _ = np.histogram2d(
        low["own_lateral"], low["own_depth"], bins=bins, range=[xlim, ylim], density=True
    )
    return h_hi - h_lo, xedges, yedges

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, phase in zip(axes, ["early", "late"]):
    diff, xe, ye = rank_density_grid(l1_plot, phase)
    vmax = np.nanpercentile(np.abs(diff), 99)
    im = ax.imshow(
        diff.T, origin="lower", aspect="auto",
        extent=[xe[0], xe[-1], ye[0], ye[-1]],
        cmap="RdBu_r", vmin=-vmax, vmax=vmax
    )
    ax.axhline(0.5, color="gray", ls="--", lw=0.7, alpha=0.6)
    ax.axvline(0, color="gray", ls="--", lw=0.7, alpha=0.6)
    ax.set_title(f"{phase}: high − low density")
    ax.set_xlabel("own_lateral")
    ax.set_ylabel("own_depth")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Where high ranks spend MORE time (red) vs less (blue)", y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_paths_density_diff.png", dpi=170, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_paths_density_diff.png")
print("Red = high ranks relatively more often there. Blue = low ranks more often there.")


In [ ]:
# Example trajectories: a few high vs low players over a full match
# (not averages — actual paths, so you can see shape differences)

def pick_example_players(df, skill, n=4, min_points=80):
    sub = df[df["indiv_label"] == skill]
    counts = sub.groupby(["match_id", "player_slot"]).size()
    counts = counts[counts >= min_points]
    if len(counts) == 0:
        return []
    picks = counts.sample(min(n, len(counts)), random_state=42 if skill == "low" else 7)
    return list(picks.index)

fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=True, sharey=True)
for row, skill in enumerate(["low", "high"]):
    picks = pick_example_players(l1_plot, skill, n=4)
    for col, key in enumerate(picks):
        ax = axes[row, col]
        mid, slot = key
        traj = l1_plot[(l1_plot["match_id"] == mid) & (l1_plot["player_slot"] == slot)].sort_values("sec")
        ax.plot(traj["own_lateral"], traj["own_depth"], lw=1.0, alpha=0.85,
                color="#4C72B0" if skill == "low" else "#DD8452")
        ax.scatter(traj["own_lateral"].iloc[0], traj["own_depth"].iloc[0], s=20, c="green", zorder=3, label="start")
        ax.scatter(traj["own_lateral"].iloc[-1], traj["own_depth"].iloc[-1], s=20, c="red", zorder=3, label="end")
        ax.set_xlim(-9000, 9000)
        ax.set_ylim(-0.05, 1.05)
        ax.axhline(0.5, color="gray", ls="--", lw=0.6, alpha=0.5)
        ax.axvline(0, color="gray", ls="--", lw=0.6, alpha=0.5)
        ax.set_title(f"{skill} | match {mid}\nslot {slot}", fontsize=9)
        if row == 1:
            ax.set_xlabel("own_lateral")
        if col == 0:
            ax.set_ylabel("own_depth")

axes[0, 0].legend(loc="upper right", fontsize=7, frameon=False)
fig.suptitle("Example full-match trajectories (team-relative map)", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_example_trajectories.png", dpi=170, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_example_trajectories.png")
print("Green dot = start, red dot = end. These are individual stories, not averages.")


In [ ]:
# Early-game only paths (first third): rank differences show up more in laning sometimes
early = l1_plot[l1_plot["phase_third"] == "early"].copy() if "phase_third" in l1_plot.columns else l1_plot

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, skill, color in zip(axes, ["low", "high"], ["#4C72B0", "#DD8452"]):
    # sample a manageable number of player-match trajectories
    keys = early.groupby(["match_id", "player_slot"]).size()
    keys = keys[keys >= 20].sample(min(40, (keys >= 20).sum()), random_state=1 if skill == "low" else 2)
    for (mid, slot) in keys.index:
        # only plot this skill's players
        row0 = early[(early["match_id"] == mid) & (early["player_slot"] == slot)]
        if row0.empty or row0["indiv_label"].iloc[0] != skill:
            continue
        traj = row0.sort_values("sec")
        ax.plot(traj["own_lateral"], traj["own_depth"], color=color, alpha=0.25, lw=0.9)
    ax.set_title(f"{skill} ranks — early paths (many players overlaid)")
    ax.set_xlim(-9000, 9000)
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color="gray", ls="--", lw=0.7, alpha=0.5)
    ax.axvline(0, color="gray", ls="--", lw=0.7, alpha=0.5)
    ax.set_xlabel("own_lateral")
axes[0].set_ylabel("own_depth")
fig.suptitle("Early-game path bundles by rank", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_early_path_bundles.png", dpi=170, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "fig_early_path_bundles.png")


In [ ]:
print("Section 13 outputs in", OUT_DIR)
print("  fig_feature_separation.png      — which L2 features separate rank")
print("  fig_paths_density_early.png     — map occupancy early")
print("  fig_paths_density_late.png      — map occupancy late")
print("  fig_paths_density_diff.png      — high−low density difference")
print("  fig_example_trajectories.png    — individual path examples")
print("  fig_early_path_bundles.png      — overlaid early paths by rank")
print("\nIf density-diff is mostly noise and path bundles look similar,")
print("that supports the 'small average differences' finding — still a valid result.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style="whitegrid")

# Time-thirds × Lane Occupancy Share Heatmap

live_l1 = l1[~l1["dead_mask"]].copy()
live_l1["phase_third"] = pd.Categorical(live_l1["phase_third"], categories=["early", "mid", "late"], ordered=True)

# 1. Calculate occupancy
occupancy = live_l1.groupby(['skill_label', 'phase_third', 'zone'], observed=True).size().reset_index(name='counts')

# 2. Calculate share of occupancy
phase_totals = occupancy.groupby(['skill_label', 'phase_third'], observed=True)['counts'].transform('sum')
occupancy['share'] = occupancy['counts'] / phase_totals

# 3. Pivot data for heatmap
pivot_low = occupancy[occupancy['skill_label'] == 'low'].pivot(index='zone', columns='phase_third', values='share')
pivot_high = occupancy[occupancy['skill_label'] == 'high'].pivot(index='zone', columns='phase_third', values='share')

# for colormap scale
vmax_val = occupancy['share'].max()

# 4. visualization using Subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("Lane Occupancy Share Heatmap by Time-thirds", fontsize=16, fontweight='bold', y=1.05)

# Low rank (blue)
sns.heatmap(pivot_low, annot=True, fmt=".1%", cmap="Blues", ax=axes[0],
            vmin=0, vmax=vmax_val, cbar_kws={'label': 'Occupancy Share'})
axes[0].set_title("Low Rank", fontsize=14)
axes[0].set_ylabel("Map Zone (Lane)", fontsize=12)
axes[0].set_xlabel("Match Phase", fontsize=12)

# high rank (red/orange)
sns.heatmap(pivot_high, annot=True, fmt=".1%", cmap="Oranges", ax=axes[1],
            vmin=0, vmax=vmax_val, cbar_kws={'label': 'Occupancy Share'})
axes[1].set_title("High Rank", fontsize=14)
axes[1].set_ylabel("")
axes[1].set_xlabel("Match Phase", fontsize=12)

# y-axis text alignment
axes[0].set_yticklabels([label.get_text().capitalize() for label in axes[0].get_yticklabels()], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="white")

# Correlation Heatmap of Movement & Spatial Variables

# 1. Selelct numeric variables for corr. analysis
numeric_cols = live_l1.select_dtypes(include=[np.number]).columns.tolist()

# 2. seperate data and calc pearson coeff
corr_low = live_l1[live_l1['skill_label'] == 'low'][numeric_cols].corr()
corr_high = live_l1[live_l1['skill_label'] == 'high'][numeric_cols].corr()

# 3. For better visualization, create mask for upper triangle of heatmap
mask_low = np.triu(np.ones_like(corr_low, dtype=bool))
mask_high = np.triu(np.ones_like(corr_high, dtype=bool))

# 4. Visualziation
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Correlation Matrix of Spatial Variables: Low vs High Rank", fontsize=16, fontweight='bold', y=1.02)

# colormap settings
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# low rank
sns.heatmap(
    corr_low, mask=mask_low, cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt=".2f", square=True, linewidths=.5,
    cbar_kws={"shrink": .8, "label": "Pearson Correlation"}, ax=axes[0]
)
axes[0].set_title("Low Rank", fontsize=14)

# high rank
sns.heatmap(
    corr_high, mask=mask_high, cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt=".2f", square=True, linewidths=.5,
    cbar_kws={"shrink": .8, "label": "Pearson Correlation"}, ax=axes[1]
)
axes[1].set_title("High Rank", fontsize=14)

plt.tight_layout()
plt.show()

**Report Visualizations**

Avter getting a better insight into the differences between the players movements across skill levels, these are the visualizations we chose to use in the report.


In [ ]:
"""
SIADS 699 — Report visualizations
Run after you have:
  - positions (or a live subset with own_depth, own_lateral, sec, match_id, player_slot)
  - matches with skill_label, duration_s
  - players with rank_tier

Saves PNGs into OUT_DIR.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- style----------
plt.rcParams.update({
    "font.size": 11,          # No one set it less than 8!
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

OUT_DIR = Path("report_figure")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOW_C, HIGH_C = "#4C72B0", "#DD8452"
PHASES = ["early", "mid", "late"]
LANES = ["top", "mid", "bottom"]


# ---------- helpers ----------
def assign_phase_third(sec, duration):
    if duration is None or not np.isfinite(duration) or duration <= 0:
        return "mid"
    if sec < duration / 3:
        return "early"
    if sec < 2 * duration / 3:
        return "mid"
    return "late"


def prepare_live(positions, matches, players):
    """
    Expects team-relative columns already present: own_depth, own_lateral.
    If not, run to_team_frame(positions) first.
    """
    pos = positions.copy()
    dur = matches.set_index("match_id")["duration_s"].to_dict()
    md = pos["match_id"].map(dur).to_numpy(dtype=float)
    sec = pos["sec"].to_numpy()
    phase = np.full(len(pos), "mid", dtype=object)
    valid = np.isfinite(md) & (md > 0)
    phase[valid & (sec < md / 3)] = "early"
    phase[valid & (sec >= 2 * md / 3)] = "late"
    pos["phase_third"] = phase
    pos["zone"] = pd.cut(
        pos["own_lateral"],
        bins=[-np.inf, -2000, 2000, np.inf],
        labels=["top", "mid", "bottom"],
    )
    pos = pos.merge(matches[["match_id", "skill_label"]], on="match_id", how="left")
    pos = pos.merge(
        players[["match_id", "player_slot", "rank_tier"]],
        on=["match_id", "player_slot"],
        how="left",
    )
    # drop near-fountain / likely dead samples for movement stats
    live = pos[pos["own_depth"] >= 0.05].copy()
    live = live.sort_values(["match_id", "player_slot", "sec"])
    g = live.groupby(["match_id", "player_slot"], sort=False)
    live["speed"] = (
        np.sqrt(g["own_depth"].diff() ** 2 + g["own_lateral"].diff() ** 2)
        / g["sec"].diff().fillna(2.0).clip(lower=0.5)
    )
    live["is_idle"] = live["speed"] < 50
    return live


# live = prepare_live(positions, matches, players)


# Figure 1 — Lane occupancy

def fig1_lane_occupancy(live, out=OUT_DIR / "fig1_lane_occupancy.png"):
    ts = (
        live.groupby(["skill_label", "phase_third", "zone"], observed=True)
        .size()
        .rename("n")
        .reset_index()
    )
    ts["share"] = ts.groupby(["skill_label", "phase_third"])["n"].transform(
        lambda s: s / s.sum()
    )

    x = np.arange(len(LANES))
    width = 0.35
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)

    for ax, phase in zip(axes, PHASES):
        for i, (skill, color) in enumerate([("low", LOW_C), ("high", HIGH_C)]):
            sub = (
                ts[(ts["phase_third"] == phase) & (ts["skill_label"] == skill)]
                .set_index("zone")
                .reindex(LANES)
                .fillna(0)
            )
            ax.bar(x + i * width, sub["share"].values, width=width, label=skill, color=color)
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels([l.capitalize() for l in LANES])
        ax.set_title(f"{phase.capitalize()} third")
        ax.set_ylim(0, 0.65)
        if ax is axes[0]:
            ax.set_ylabel("Share of live time in lane")
            ax.legend(frameon=False, title="Lobby skill")

    fig.suptitle("Lane occupancy is similar for high and low rank lobbies", y=1.03)
    fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches="tight")  # bbox keeps labels from clipping
    plt.show()
    print("Saved", out)

# Figure 2 — Metrics over match thirds

def fig2_metrics_over_time(live, out=OUT_DIR / "fig2_metrics_over_time.png"):
    ov = (
        live.groupby(["skill_label", "phase_third"])
        .agg(
            mean_depth=("own_depth", "mean"),
            idle_share=("is_idle", "mean"),
            mean_speed=("speed", "mean"),
        )
        .reset_index()
    )

    metrics = [
        ("mean_depth", "Mean depth (aggression)", "own_depth (0=fountain → 1=enemy)"),
        ("idle_share", "Idle share", "Share of time with speed < 50"),
        ("mean_speed", "Mean speed", "Speed (team-relative units / sec)"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))

    for ax, (col, title, ylabel) in zip(axes, metrics):
        for skill, color in [("low", LOW_C), ("high", HIGH_C)]:
            s = (
                ov[ov["skill_label"] == skill]
                .set_index("phase_third")
                .reindex(PHASES)
            )
            ax.plot(PHASES, s[col], marker="o", lw=2, color=color, label=skill)
        ax.set_title(title)
        ax.set_xlabel("Match third")
        ax.set_ylabel(ylabel)
        if ax is axes[0]:
            ax.legend(frameon=False, title="Lobby skill")

    fig.suptitle("Higher-rank lobbies move a bit more and idle a bit less", y=1.03)
    fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved", out)

# Figure 3 — Early map density

def fig3_density_early(live, out=OUT_DIR / "fig3_density_early.png", max_points=50000):
    early = live[live["phase_third"] == "early"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

    for ax, skill, cmap in zip(axes, ["low", "high"], ["Blues", "Oranges"]):
        s = early[early["skill_label"] == skill]
        if len(s) > max_points:
            s = s.sample(max_points, random_state=42)
        hb = ax.hexbin(
            s["own_lateral"], s["own_depth"],
            gridsize=40, cmap=cmap, mincnt=2, bins="log",
        )
        ax.axhline(0.5, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.set_title(f"{skill.capitalize()} lobbies — early game")
        ax.set_xlabel("own_lateral  (← top lane | bottom →)")
        ax.set_xlim(-9000, 9000)
        ax.set_ylim(-0.02, 1.02)
        fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)

    axes[0].set_ylabel("own_depth  (0 = own fountain, 1 = enemy)")
    fig.suptitle("Early-game map occupancy by lobby skill", y=1.02)
    fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved", out)

# Figure 4 — Rank overlap inside match labels

def fig4_rank_overlap(players, matches, out=OUT_DIR / "fig4_rank_overlap.png"):
    r = players.merge(matches[["match_id", "skill_label"]], on="match_id", how="inner")
    r = r.dropna(subset=["rank_tier", "skill_label"])

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for skill, color in [("low", LOW_C), ("high", HIGH_C)]:
        vals = r.loc[r["skill_label"] == skill, "rank_tier"]
        ax.hist(vals, bins=20, density=True, alpha=0.55, label=skill, color=color)
    ax.set_xlabel("Individual player rank_tier")
    ax.set_ylabel("Density")
    ax.set_title("Individual ranks still overlap inside high vs low match labels")
    ax.legend(frameon=False, title="Match skill_label")
    fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved", out)

In [ ]:
# Where files  go
OUT_DIR = Path("/content/report_figure")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving figures to:", OUT_DIR.resolve())

# Positions are in team-relative frame
if "own_depth" not in positions.columns:
    positions = to_team_frame(positions)

# Build the live table
live = prepare_live(positions, matches, players)
print("live rows:", len(live))
print(live["skill_label"].value_counts(dropna=False))

# Ggenerate the four report figures
fig1_lane_occupancy(live, out=OUT_DIR / "fig1_lane_occupancy.png")
fig2_metrics_over_time(live, out=OUT_DIR / "fig2_metrics_over_time.png")
fig3_density_early(live, out=OUT_DIR / "fig3_density_early.png")
fig4_rank_overlap(players, matches, out=OUT_DIR / "fig4_rank_overlap.png")

print("Files written:")
for p in sorted(OUT_DIR.glob("*.png")):
    print(" ", p, p.stat().st_size, "bytes")